In [1]:
import pandas as pd
import numpy as np

In [2]:
ratings_full = pd.read_csv('../data/processed/user_item_ratings_subrating.csv')
user_df = pd.read_csv('../data/processed/user_features.csv')
item_df = pd.read_csv('../data/processed/item_features.csv')

print("Data simulation complete.")
print("\nRatings Full DF shape:", ratings_full.shape)
print("\nItem Features DF shape:", item_df.shape)
print("\nUser Features DF shape:", user_df.shape)
ratings_full

Data simulation complete.

Ratings Full DF shape: (1400102, 9)

Item Features DF shape: (27197, 9)

User Features DF shape: (46660, 9)


,user_id,item_id,rating,sleep quality,value,rooms,service,cleanliness,location
0,1521,19,5.0,NaN,5.0,NaN,5.0,5.0,NaN
1,1524,19,5.0,NaN,NaN,5.0,5.0,NaN,3.0
2,1527,19,5.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1529,19,4.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1530,19,4.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1400097,613882,365056,5.0,NaN,4.0,5.0,NaN,5.0,5.0
1400098,1521429,365056,2.0,NaN,3.0,2.0,3.0,2.0,1.0
1400099,76310,365056,4.0,NaN,NaN,NaN,NaN,NaN,NaN
1400100,272435,365056,3.0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# --- 1. Create the Popularity-Based Model ---

# Calculate average rating and number of ratings for each item
item_stats = ratings_full.groupby('item_id')['rating'].agg(['mean', 'count']).reset_index()
item_stats.rename(columns={'mean': 'avg_rating', 'count': 'num_ratings'}, inplace=True)

# Calculate a simple popularity score
# Here, we give a weight to the number of ratings to penalize items with few ratings.
# The weight can be tuned based on the distribution of num_ratings.
# For this example, we'll use a simple addition, but more complex formulas can be used.
C = item_stats['avg_rating'].mean()
m = item_stats['num_ratings'].quantile(0.90) # Use the 90th percentile as the minimum votes required

def weighted_rating(x, m=m, C=C):
    v = x['num_ratings']
    R = x['avg_rating']
    return (v / (v + m) * R) + (m / (v + m) * C)

item_stats['popularity_score'] = item_stats.apply(weighted_rating, axis=1)


# Get the ranked list of all items
popular_items = item_stats.sort_values('popularity_score', ascending=False)

print("Top 10 most popular items:")
print(popular_items.head(10))
print("\n")


Top 10 most popular items:
       item_id  avg_rating  num_ratings  popularity_score
9762     51082    4.712000          875          4.645464
9691     51001    4.754941          506          4.640749
9445     50695    4.786744          347          4.624566
25406   362561    4.775281          356          4.618728
24344   361296    4.838843          242          4.611370
9804     51137    4.747500          400          4.610160
25302   362443    4.734644          407          4.601618
24831   361891    4.790780          282          4.599491
24564   361580    4.805344          262          4.599462
26149   363417    4.771523          302          4.594453




In [4]:
# --- 2. Prepare Data for Evaluation ---

# To evaluate, we need to compare the recommendations against a user's actual liked items.
# We'll consider items with a rating of 4 or 5 as "relevant" or "liked".
test_user_items = ratings_full[ratings_full['rating'] >= 4]
test_user_items = test_user_items.groupby('user_id')['item_id'].apply(list).to_dict()

# Get a list of all unique users in the test set
all_users = list(test_user_items.keys())


# --- 3. Define Evaluation Metrics ---

def precision_at_k(recommended_items, relevant_items, k):
    recommended_at_k = recommended_items[:k]
    num_relevant_and_recommended = len(set(recommended_at_k) & set(relevant_items))
    return num_relevant_and_recommended / k

def recall_at_k(recommended_items, relevant_items, k):
    recommended_at_k = recommended_items[:k]
    num_relevant_and_recommended = len(set(recommended_at_k) & set(relevant_items))
    return num_relevant_and_recommended / len(relevant_items) if relevant_items else 0

def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asfarray(relevance_scores)[:k]
    if relevance_scores.size:
        return np.sum(relevance_scores / np.log2(np.arange(2, relevance_scores.size + 2)))
    return 0.

def ndcg_at_k(recommended_items, relevant_items, k):
    recommended_relevance = [1 if item in relevant_items else 0 for item in recommended_items]
    ideal_relevance = [1] * len(relevant_items)

    dcg = dcg_at_k(recommended_relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)

    return dcg / idcg if idcg > 0 else 0

In [5]:
# --- 4. Evaluate the Model ---

k_values = [ 10, 20, 50, 100]
evaluation_results = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in k_values}

# Get the top 50 popular items for recommendations
top_50_popular_items = popular_items['item_id'].tolist()

for user in all_users:
    # For a popularity model, the recommendations are the same for every user
    recommendations = top_50_popular_items
    relevant_user_items = test_user_items.get(user, [])

    for k in k_values:
        precision = precision_at_k(recommendations, relevant_user_items, k)
        recall = recall_at_k(recommendations, relevant_user_items, k)
        ndcg = ndcg_at_k(recommendations, relevant_user_items, k)

        evaluation_results[k]['precision'].append(precision)
        evaluation_results[k]['recall'].append(recall)
        evaluation_results[k]['ndcg'].append(ndcg)

# Calculate the average metrics across all users
average_results = {k: {} for k in k_values}
for k in k_values:
    average_results[k]['precision'] = np.mean(evaluation_results[k]['precision'])
    average_results[k]['recall'] = np.mean(evaluation_results[k]['recall'])
    average_results[k]['ndcg'] = np.mean(evaluation_results[k]['ndcg'])

print("Evaluation Results (Averaged over all users):")
results_df = pd.DataFrame(average_results).T
print(results_df)

Evaluation Results (Averaged over all users):
     precision    recall      ndcg
10    0.008232  0.003624  0.009767
20    0.007167  0.006316  0.008904
50    0.005855  0.012918  0.011481
100   0.005073  0.022429  0.015847


In [ ]:
# Calculate and visualize NDCG, hitrate, catalog_coverage, Gini_diveristy at k=[10,20,50,100]

